In [ ]:
!pip install tensorflow matplotlib numpy

In [ ]:
# Import required libraries
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# 1. Load CIFAR-10 dataset
print("Step 1: Loading CIFAR-10 dataset...")

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Convert labels to 1D
y_train = y_train.flatten()
y_test = y_test.flatten()

# 2. Resize images
print("Step 2: Resizing images...")

x_train = tf.image.resize(x_train, (96, 96))
x_test = tf.image.resize(x_test, (96, 96))

# 3. Preprocess images for MobileNetV2
x_train = tf.keras.applications.mobilenet_v2.preprocess_input(x_train)
x_test = tf.keras.applications.mobilenet_v2.preprocess_input(x_test)

# 4. Load pre-trained MobileNetV2
print("Step 3: Loading MobileNetV2 model...")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pre-trained layers
base_model.trainable = False

# 5. Create model
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

# 6. Compile model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Display model summary
model.summary()

# 7. Train model
print("\nStep 4: Training model...")

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# 8. Evaluate model
print("\nStep 5: Testing model...")

loss, accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", accuracy * 100, "%")

# 9. Predict one image
prediction = model.predict(x_test[:1])

classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

predicted_class = np.argmax(prediction)

print("\nActual:", classes[y_test[0]])
print("Predicted:", classes[predicted_class])

# 10. Print accuracy
train_accuracy = history.history["accuracy"][-1]
val_accuracy = history.history["val_accuracy"][-1]

print("\nTraining Accuracy:", train_accuracy * 100, "%")
print("Validation Accuracy:", val_accuracy * 100, "%")
print("Test Accuracy:", accuracy * 100, "%")

# 11. Display accuracy graph
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid()

plt.show()

# 12. Display predicted image
plt.figure(figsize=(4, 4))

plt.imshow(x_test[0] / 2 + 0.5)
plt.title("Predicted: " + classes[predicted_class])
plt.axis("off")

plt.show()